# Einsum playground

Last edited: 2026-10-22

**TLDR:** `einsum` lets you write a tensor operation by *naming each axis with a letter*.
Letters that appear on the left but not on the right get added up.
Letters shared by two inputs get multiplied together.

Run the cells top to bottom. Change the strings and shapes, and see what breaks.

## Preparations

In [2]:
# we need some libraries for this demo
%pip install einops numpy -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import einops

np.set_printoptions(linewidth=120)
rng = np.random.default_rng(0)

# small integer arrays are easy to check by hand
A = np.arange(6).reshape(2, 3)   # shape (2, 3)
B = np.arange(12).reshape(3, 4)  # shape (3, 4)
v = np.array([1, 2, 3])
w = np.array([10, 20, 30])
print("A =\n", A)
print("B =\n", B)

A =
 [[0 1 2]
 [3 4 5]]
B =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


## The three rules

An einsum string looks like `"inputs -> output"`, for example `"ij,jk->ik"`.

1. Each input gets one letter per axis. `"ij"` means a 2D array: axis 0 is called `i`, axis 1 is called `j`.
2. The output lists which letters survive, and in what order.
3. Any letter missing from the output is **summed over**. A letter used in several inputs means those axes are **lined up and multiplied**.

Read `"ij,jk->ik"` aloud: *for every i and k, multiply A[i, j] by B[j, k] for each j, then add over j.* That is matrix multiplication.

## 1. One input: reorder and sum

In [26]:
print(type(A))
# identity: nothing changes
print(np.einsum("ij->ij", A))


# transpose: same letters, swapped order in the output
print(np.einsum("ij->ji", A))
print(A.T)

<class 'numpy.ndarray'>
[[0 1 2]
 [3 4 5]]
[[0 3]
 [1 4]
 [2 5]]
[[0 3]
 [1 4]
 [2 5]]


In [32]:
print(A)
# sum everything: both letters are missing from the output
# print(np.einsum("ij->j", A))
# sum over j only (add up each row)
print(np.einsum("ij->i", A), A.sum(axis=0), A, A, A.sum()) # This should output

# sum over i only (add up each column)
print(np.einsum("ij->j", A))


[[0 1 2]
 [3 4 5]]
[ 3 12] [3 5 7] [[0 1 2]
 [3 4 5]] [[0 1 2]
 [3 4 5]] 15
[3 5 7]


In [35]:

S = np.arange(9).reshape(3, 3)
# print(S)
# print(np.einsum("ij->i", S)) 
print(np.einsum("ii->", S)) 

12


## 2. Two inputs: multiply, then maybe sum

In [40]:
# shared letter kept in the output -> elementwise multiply
print(np.einsum("i,i->i", v, w))

# shared letter dropped from the output -> multiply, then sum: dot product
print(np.einsum("i,i->", v, w), v @ w)

# different letters, both kept -> every pair gets multiplied: outer product
print(np.einsum("i,j->ij", v, w))
# print(np.outer(v, w))

[10 40 90]
140 140
[[10 20 30]
 [20 40 60]
 [30 60 90]]


In [41]:
# matrix multiply: j is shared and dropped
print(np.einsum("ij,jk->ik", A, B))
print(A @ B)

# matrix times vector
print(np.einsum("ij,j->i", A, v), A @ v)

[[20 23 26 29]
 [56 68 80 92]]
[[20 23 26 29]
 [56 68 80 92]]
[ 8 26] [ 8 26]


## 3. Extra leading axes (batches)

A letter that just rides along in both the inputs and the output is a batch axis.
Nothing is summed over it, so each batch item is handled separately.

In [ ]:
X = rng.normal(size=(5, 2, 3))   # 5 matrices of shape (2, 3)
Y = rng.normal(size=(5, 3, 4))   # 5 matrices of shape (3, 4)
# print(X)
# print(Y)
out = np.einsum("aij,ajk->aik", X, Y) # Performs matrix multiplication on each of the 5 matrices
print(out.shape)
# print(np.allclose(out, X @ Y))

# # the same, but one matrix B shared by every batch item (no b on B)
out2 = np.einsum("bij,jk->bik", X, B)
print(out2.shape)

(5, 2, 4)


## 4. `einops.einsum`: readable axis names and `...`

`einops.einsum` accepts whole words as axis names, and works with numpy, torch and others.
Spaces separate the names. `...` stands for any number of extra axes.

In [51]:
# note: with einops.einsum the arrays come FIRST and the pattern comes LAST
out = einops.einsum(X, Y, "batch rows inner, batch inner cols -> batch rows cols")
# print(out.shape)

# `...` matches any number of leading axes
Z = rng.normal(size=(7, 5, 2, 3))
out = einops.einsum(Z, B, "... rows inner, inner cols -> ... rows cols")
print(out.shape)

(7, 5, 2, 4)


## Exercises

Fill in each `None` with an einsum string. The `assert` compares against the plain numpy answer.
If an assert fails, print `A`, the shapes, and your string, and trace which letters get summed.

In [54]:
# Ex 1: the sum of each column of M (result shape: (4,))
M = rng.normal(size=(3, 4))
pattern =  "ij -> j" # TODO
assert np.allclose(np.einsum(pattern, M), M.sum(axis=0))

In [55]:
# Ex 2: multiply M elementwise with N, then sum the whole thing to one number
N = rng.normal(size=(3, 4))
pattern = "ij, ij ->"  # TODO
assert np.allclose(np.einsum(pattern, M, N), (M * N).sum())

In [59]:

pattern = "ij, kj ->ik "  # TODO
assert np.allclose(np.einsum(pattern, M, N), M @ N.T)

In [60]:
P = rng.normal(size=(5, 8))
Q = rng.normal(size=(5, 8))
pattern = "ij, ij-> i"  # TODO
assert np.allclose(np.einsum(pattern, P, Q), (P * Q).sum(axis=1))

In [61]:
T = rng.normal(size=(5, 3, 4))
pattern = "ijk -> ikj"  # TODO
assert np.allclose(np.einsum(pattern, T), T.transpose(0, 2, 1))

In [66]:

U = rng.normal(size=(2, 6, 3))
W = rng.normal(size=(4, 3))
pattern = "batch rows inner, cols inner -> batch rows cols" # TODO
out = einops.einsum(U, W, pattern)
assert np.allclose(out, U @ W.T)

## Things to try on your own

- Give `np.einsum` a string whose letter has two different sizes in two inputs. Read the error.
- Leave a letter out of the output that appears in only one input. What gets summed?
- Put a letter in the output that is in no input. What does numpy say?
- Rewrite each example above using `einops.einsum` with full-word axis names.